# MERGE (upsert) — notatki referencyjne (SQL Server)

Przykłady na modelu: `dim_Klienci` (ID_Klienta, Nazwa, Segment) jako tabela docelowa, `stg_Klienci` (dane przychodzące z ETL/importu) jako źródło.

## 1. Co robi `MERGE` — jedno polecenie, trzy operacje naraz

`MERGE` porównuje tabelę docelową ze źródłem i w jednym poleceniu wykonuje `INSERT` (dla wierszy nowych), `UPDATE` (dla zmienionych) i opcjonalnie `DELETE` (dla wierszy, które zniknęły ze źródła) — klasyczny "upsert" plus obsługa usunięć.

### Składnia szkieletowa

```sql
MERGE INTO <tabela_docelowa> AS target
USING <źródło> AS source
ON <warunek_dopasowania>
WHEN MATCHED THEN
    UPDATE SET ...
WHEN NOT MATCHED BY TARGET THEN
    INSERT (...) VALUES (...)
WHEN NOT MATCHED BY SOURCE THEN
    DELETE
;
```

**Trzy klauzule `WHEN`, każda opcjonalna, ale przynajmniej jedna wymagana:**
- **`WHEN MATCHED`** — wiersz istnieje w obu tabelach (dopasowany przez `ON`) → zwykle `UPDATE`, może być też `DELETE`.
- **`WHEN NOT MATCHED BY TARGET`** — wiersz jest w źródle, ale nie w celu → zwykle `INSERT` (nowy rekord).
- **`WHEN NOT MATCHED BY SOURCE`** — wiersz jest w celu, ale zniknął ze źródła → zwykle `DELETE` (albo `UPDATE`, np. oznaczenie jako nieaktywny).

### Przykład — upsert klientów z tabeli stagingowej

```sql
MERGE INTO dim_Klienci AS target
USING stg_Klienci AS source
ON target.ID_Klienta = source.ID_Klienta
WHEN MATCHED AND (target.Nazwa <> source.Nazwa OR target.Segment <> source.Segment) THEN
    UPDATE SET target.Nazwa = source.Nazwa, target.Segment = source.Segment
WHEN NOT MATCHED BY TARGET THEN
    INSERT (ID_Klienta, Nazwa, Segment) VALUES (source.ID_Klienta, source.Nazwa, source.Segment)
;
```

**Zwróć uwagę na dodatkowy warunek w `WHEN MATCHED AND (...)`** — bez niego `UPDATE` wykonałby się dla **każdego** dopasowanego wiersza, nawet jeśli nic się nie zmieniło (zbędny zapis, niepotrzebnie odświeża statystyki/znaczniki czasu modyfikacji, generuje ruch w logu transakcyjnym). Warunek dodatkowy ogranicza `UPDATE` tylko do faktycznie zmienionych wierszy — dokładnie ta sama zasada co porównanie przez `HASHBYTES` w widoku SCD2, o którym rozmawialiśmy wcześniej.

## 2. Krytyczne ostrzeżenie — `MERGE` ma udokumentowaną historię błędów i problemów z współbieżnością

To jest najważniejsza sekcja tego pliku, ważniejsza niż sama składnia. `MERGE` w SQL Server (wprowadzony w 2008) ma **udokumentowaną, wieloletnią historię realnych problemów** — nie plotki, tylko konkretne, opisane przez ekspertów społeczności SQL Server (Aaron Bertrand, Michael J. Swart) przypadki:

### Problem 1 — pozorna atomowość, realny brak atomowości

`MERGE` **wygląda** jak jedna, atomowa operacja, ale pod spodem silnik wykonuje operacje `INSERT`/`UPDATE`/`DELETE` **niezależnie od siebie**. Przy równoczesnym wykonaniu tego samego `MERGE` przez wiele sesji/wątków naraz (typowy scenariusz w procesach ETL uruchamianych równolegle, albo w aplikacji z wieloma użytkownikami zapisującymi jednocześnie) realne jest ryzyko **race condition** — dwie sesje mogą "jednocześnie" uznać, że dany wiersz nie istnieje (obie widzą stan sprzed zapisu drugiej), obie próbują `INSERT`, i jedna z nich dostaje błąd naruszenia klucza głównego, mimo że logicznie `MERGE` miał to obsłużyć bezpiecznie.

**Rozwiązanie — hint `WITH (HOLDLOCK)` na tabeli docelowej:**

```sql
MERGE INTO dim_Klienci WITH (HOLDLOCK) AS target
USING stg_Klienci AS source
ON target.ID_Klienta = source.ID_Klienta
...
```

`HOLDLOCK` wymusza poziom izolacji `SERIALIZABLE` na tabeli docelowej dla czasu trwania `MERGE` — eliminuje okno czasowe, w którym dwie równoległe sesje mogłyby "zobaczyć" ten sam brak wiersza i obie spróbować go wstawić. **Praktyczna obserwacja z materiałów źródłowych: ten hint jest notorycznie pomijany w kodzie produkcyjnym** — większość przykładów `MERGE`, które zobaczysz w internecie (włącznie z oficjalną dokumentacją Microsoftu), go nie zawiera, mimo że eksperci uznają go za praktycznie obowiązkowy przy jakiejkolwiek współbieżności.

### Problem 2 — udokumentowane błędy silnika w starszych wersjach

Społeczność SQL Server udokumentowała rzeczywiste błędy w implementacji `MERGE` (naruszenia unikalności, problemy z indeksami filtrowanymi, zakleszczenia, błędy asercji wewnętrznej silnika) w różnych wersjach SQL Server na przestrzeni lat. Część z nich została naprawiona w nowszych wersjach/aktualizacjach, część wymaga obejść (trace flags). **To jest jedna z niewielu instrukcji T-SQL, przy której doświadczeni administratorzy baz danych otwarcie zalecają rozważenie unikania jej** na rzecz osobnych `INSERT`/`UPDATE`, szczególnie w środowiskach o wysokiej współbieżności zapisu.

### Kontrargument, dla uczciwego obrazu

Nie wszyscy eksperci są zgodni — część praktyków (w tym autorzy niektórych z cytowanych analiz) używa `MERGE` bez problemów w scenariuszach **hurtowni danych z kontrolowanym, sekwencyjnym procesem ETL** (jeden proces ładujący dane w danym oknie czasowym, bez współbieżnych zapisów do tej samej tabeli) — to jest dokładnie Twój typowy scenariusz (ETL/ładowanie danych kadrowych/sprzedażowych, nie wielu użytkowników zapisujących jednocześnie do tej samej tabeli faktów). **W takim kontrolowanym, jednowątkowym scenariuszu ryzyko z Problemu 1 praktycznie nie występuje** (nie ma współbieżności do zabezpieczenia), a Problem 2 dotyczy głównie bardziej złożonych konfiguracji (indeksy filtrowane, partycjonowanie, triggery).

**Praktyczna rekomendacja dla Twojego workflow:** jeśli `MERGE` uruchamiasz w kontrolowanym procesie ETL (jeden proces na raz, bez współbieżnych zapisów do tej samej tabeli z innych źródeł) — **MERGE jest bezpieczny i wygodny**, dopisz `WITH (HOLDLOCK)` z przyzwyczajenia (nic nie kosztuje w scenariuszu bez współbieżności, chroni na wypadek, gdyby to założenie kiedyś przestało być prawdziwe). Jeśli projektujesz coś, co może być wywoływane współbieżnie (np. z wielu równoległych tasków Airflow zapisujących do tej samej tabeli) — rozważ poważnie osobne `INSERT`/`UPDATE` z jawną logiką transakcyjną, albo koniecznie przetestuj `MERGE` pod obciążeniem współbieżnym przed wdrożeniem.

## 3. `WHEN NOT MATCHED BY SOURCE` — obsługa usunięć, i dlaczego to najbardziej ryzykowna klauzula

```sql
MERGE INTO dim_Klienci WITH (HOLDLOCK) AS target
USING stg_Klienci AS source
ON target.ID_Klienta = source.ID_Klienta
WHEN MATCHED AND (target.Nazwa <> source.Nazwa) THEN
    UPDATE SET target.Nazwa = source.Nazwa
WHEN NOT MATCHED BY TARGET THEN
    INSERT (ID_Klienta, Nazwa) VALUES (source.ID_Klienta, source.Nazwa)
WHEN NOT MATCHED BY SOURCE THEN
    DELETE
;
```

**Realne, częste niebezpieczeństwo:** jeśli `stg_Klienci` (źródło) z jakiegokolwiek powodu jest **niekompletne** przy danym uruchomieniu (np. błąd w procesie poprzedzającym, źródłowy plik dotarł tylko częściowo, filtr WHERE przypadkiem ograniczył dane) — `WHEN NOT MATCHED BY SOURCE THEN DELETE` **usunie z tabeli docelowej wszystkich klientów, których zabrakło w tym niekompletnym imporcie**, mimo że w rzeczywistości nadal istnieją. To jest jeden z najczęstszych, najbardziej kosztownych błędów przy pracy z `MERGE` — przypadkowe hurtowe usunięcie danych przez niekompletne źródło.

**Bezpieczniejsza alternatywa — `UPDATE` zamiast `DELETE`, oznaczenie jako nieaktywny:**

```sql
WHEN NOT MATCHED BY SOURCE THEN
    UPDATE SET target.IsActive = 0, target.DataDezaktywacji = GETDATE()
```

To jest wzorzec "soft delete" — bezpieczniejszy w praktyce, bo błędne/niekompletne źródło co najwyżej błędnie oznaczy rekordy jako nieaktywne (odwracalne), zamiast trwale je usunąć.

**Jeszcze bezpieczniejsze — ogranicz zasięg `MERGE` jawnym warunkiem, zamiast operować na całej tabeli docelowej:**

```sql
MERGE INTO dim_Klienci WITH (HOLDLOCK) AS target
USING stg_Klienci AS source
ON target.ID_Klienta = source.ID_Klienta
    AND target.ID_Placowki = @PlacowkaBiezacaImportu   -- ogranicza WHEN NOT MATCHED BY SOURCE tylko do tej placówki
WHEN NOT MATCHED BY SOURCE THEN
    DELETE
;
```

Jeśli import dotyczy tylko jednej placówki na raz, warunek `ON` ograniczony do tej placówki sprawia, że `WHEN NOT MATCHED BY SOURCE` "widzi" tylko wiersze tej placówki w tabeli docelowej — reszta danych (inne placówki) jest całkowicie niedotknięta, nawet jeśli źródło jest niekompletne.

## 4. `OUTPUT` — co faktycznie się zmieniło

`MERGE` może zwrócić informację o wykonanych operacjach przez klauzulę `OUTPUT` — przydatne do logowania/audytu, albo do przekazania wyniku dalej (np. do tabeli logu ETL, albo z powrotem do Pythona przez `mssql_python`).

```sql
MERGE INTO dim_Klienci WITH (HOLDLOCK) AS target
USING stg_Klienci AS source
ON target.ID_Klienta = source.ID_Klienta
WHEN MATCHED AND (target.Nazwa <> source.Nazwa) THEN
    UPDATE SET target.Nazwa = source.Nazwa
WHEN NOT MATCHED BY TARGET THEN
    INSERT (ID_Klienta, Nazwa) VALUES (source.ID_Klienta, source.Nazwa)
OUTPUT
    $action AS Operacja,          -- 'INSERT', 'UPDATE' lub 'DELETE'
    inserted.ID_Klienta,
    deleted.Nazwa AS NazwaPoprzednia,
    inserted.Nazwa AS NazwaNowa;
```

`$action` to specjalna kolumna dostępna tylko w `OUTPUT` przy `MERGE`, zwracająca tekstowo, która operacja faktycznie zaszła dla danego wiersza. `inserted`/`deleted` działają jak w standardowych triggerach — `deleted` to stan **przed** zmianą, `inserted` to stan **po**. To pozwala zbudować pełny log "co się zmieniło przy tym uruchomieniu" jednym poleceniem, bez dodatkowych zapytań porównujących stan przed/po ręcznie.

## 5. Alternatywa bez `MERGE` — osobne `UPDATE`+`INSERT`, i jej własna pułapka

Skoro `MERGE` ma udokumentowane ryzyka, warto znać bezpieczny wzorzec zastępczy:

```sql
BEGIN TRANSACTION;

UPDATE target
SET target.Nazwa = source.Nazwa, target.Segment = source.Segment
FROM dim_Klienci AS target
JOIN stg_Klienci AS source ON target.ID_Klienta = source.ID_Klienta
WHERE target.Nazwa <> source.Nazwa OR target.Segment <> source.Segment;

INSERT INTO dim_Klienci (ID_Klienta, Nazwa, Segment)
SELECT source.ID_Klienta, source.Nazwa, source.Segment
FROM stg_Klienci AS source
WHERE NOT EXISTS (
    SELECT 1 FROM dim_Klienci target WHERE target.ID_Klienta = source.ID_Klienta
);

COMMIT TRANSACTION;
```

**Ważne — to NIE jest automatycznie bezpieczniejsze bez własnej ostrożności:** klasyczny wzorzec "sprawdź czy istnieje, potem wstaw albo zaktualizuj" (tu rozdzielony na `UPDATE` + `INSERT ... WHERE NOT EXISTS`) ma **swoją własną, analogiczną** pułapkę współbieżności — między sprawdzeniem `NOT EXISTS` a wykonaniem `INSERT` inna sesja mogłaby wstawić ten sam wiersz. To jest dokładnie ten sam fundamentalny problem co przy `MERGE` bez `HOLDLOCK` — różnica jest w tym, że tu masz **pełną, jawną kontrolę** nad transakcją i poziomem izolacji (`BEGIN TRANSACTION` + odpowiedni `SET TRANSACTION ISOLATION LEVEL`), zamiast polegać na tym, co robi silnik "pod maską" przy `MERGE`.

**Uczciwe podsumowanie tej sekcji:** przejście z `MERGE` na osobne `UPDATE`+`INSERT` **nie eliminuje** problemu współbieżności samo w sobie — eliminuje tylko udokumentowane, historyczne błędy silnika specyficzne dla implementacji `MERGE`. Jeśli Twój problem to konkretnie współbieżność, potrzebujesz odpowiedniego poziomu izolacji transakcji niezależnie od tego, którą składnię wybierzesz.

## 6. Podsumowanie — kiedy `MERGE`, kiedy alternatywa

| Sytuacja | Rekomendacja |
|---|---|
| Kontrolowany, sekwencyjny proces ETL (jeden proces na raz) | `MERGE WITH (HOLDLOCK)` — bezpieczne, wygodne, czytelne w jednym miejscu |
| Możliwa współbieżność (wiele równoległych procesów zapisujących do tej samej tabeli) | Rozważ osobne `UPDATE`+`INSERT` w jawnej transakcji z odpowiednim poziomem izolacji, albo dokładnie przetestuj `MERGE` pod obciążeniem |
| `WHEN NOT MATCHED BY SOURCE` z `DELETE` | Zawsze rozważ, czy źródło może być niekompletne — ogranicz zasięg `ON` albo zamień na "soft delete" (`UPDATE ... SET IsActive = 0`) |
| Potrzebujesz logu "co się zmieniło" | `OUTPUT $action, inserted.*, deleted.*` |
| Starsza wersja SQL Server, złożona konfiguracja (indeksy filtrowane, partycje, triggery) | Sprawdź udokumentowane błędy dla Twojej konkretnej wersji przed produkcyjnym wdrożeniem `MERGE` |
| Prosty upsert, brak współbieżności, chcesz jednego czytelnego polecenia | `MERGE WITH (HOLDLOCK)` — nie ma powodu komplikować |

**Zasada nadrzędna: `HOLDLOCK` nie jest opcjonalnym dodatkiem stylistycznym — traktuj go jako domyślną część każdego `MERGE`, który piszesz, niezależnie od tego, czy w danym momencie spodziewasz się współbieżności.** Koszt dodania go jest znikomy, koszt jego braku przy nieoczekiwanej współbieżności bywa poważny.